# Fine-Tune Gemma 4 E2B-IT (QLoRA) — Robust Version

### Features
- **Stage-by-stage HF uploads** — progress is never lost between sessions
- **Cross-session resume** — training checkpoints backed up to HF Hub
- **RAM-backed merge** — merged model lives in `/dev/shm`, never touches disk quota
- **Disk guards** — each stage checks free space before proceeding
- **Smart skipping** — each stage checks HF Hub and skips if already done

### Repos used
- `ADAPTER_REPO` — stores LoRA adapter + training checkpoints (private)
- `GGUF_REPO` — stores final Q4_K_M GGUF (your existing repo)

### Workflow
```
Cell 1: Install
Cell 2: Config + Auth + Helpers
Cell 3: STAGE 1 — Train          → saves adapter to ADAPTER_REPO
Cell 4: STAGE 2 — Merge to RAM   → merged model lives in /dev/shm
Cell 5: STAGE 3 — Convert + Quant → FP16 GGUF → Q4_K_M GGUF
Cell 6: STAGE 4 — Upload GGUF    → uploads to GGUF_REPO
```
Run all cells top to bottom. If it crashes, just run all again — each stage will skip or resume automatically.

In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════════
import subprocess
subprocess.run(['pip', 'uninstall', '-y', 'torchao', 'torchvision'], capture_output=True)
subprocess.run([
    'pip', 'install', '-q', '-U',
    'git+https://github.com/huggingface/transformers.git',
    'peft', 'bitsandbytes', 'accelerate', 'datasets', 'huggingface_hub'
], check=True)
print('✅ Dependencies installed.')

✅ Dependencies installed.


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — CONFIG, AUTH & HELPERS
# Edit the USER CONFIG section, then run.
# ═══════════════════════════════════════════════════════════════
import os, gc, glob, shutil, torch, json
from huggingface_hub import (
    login, HfApi, snapshot_download, list_repo_files
)
from kaggle_secrets import UserSecretsClient

# ─── USER CONFIG — edit these ─────────────────────────────────
BASE_MODEL_ID = 'google/gemma-4-E2B-it'
ADAPTER_REPO  = 'khedim/NLP-MINI-PROJECT-adapter'   # LoRA backup (will be auto-created)
GGUF_REPO     = 'khedim/NLP-MINI-PROJECT'            # final GGUF destination
DATASET_ID    = 'ajibawa-2023/Children-Stories-Collection'
SAMPLE_SIZE   = 10000
MAX_SEQ_LEN   = 256
# ──────────────────────────────────────────────────────────────

BASE_DIR   = '/kaggle/working'
MODEL_OUT  = f'{BASE_DIR}/lora_adapter'
# MERGED_DIR = '/dev/shm/gemma_merged'      # RAM-backed, counts against RAM not disk
MERGED_DIR = f'{BASE_DIR}/gemma_merged'
FP16_GGUF  = f'{BASE_DIR}/gemma-e2b-fp16.gguf'
Q4_GGUF    = f'{BASE_DIR}/gemma-e2b-Q4_K_M.gguf'
HF_CACHE   = f'{BASE_DIR}/hf_cache'
LLAMA_DIR  = f'{BASE_DIR}/llama.cpp'

os.environ.update({
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'WANDB_DISABLED':           'true',
    'HF_HOME':                  HF_CACHE,
})
os.makedirs(MODEL_OUT, exist_ok=True)

# ─── AUTH ─────────────────────────────────────────────────────
try:
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('✅ Logged in to HuggingFace')
except Exception as e:
    raise RuntimeError(f'❌ HF_TOKEN not found in Kaggle Secrets: {e}')

api = HfApi()

# ─── HELPERS ──────────────────────────────────────────────────
def disk_free_gb(path='/kaggle/working'):
    return shutil.disk_usage(path).free / 1e9

def shm_free_gb():
    return shutil.disk_usage('/dev/shm').free / 1e9

def guard_disk(min_gb=2.5, label=''):
    free = disk_free_gb()
    tag = f' [{label}]' if label else ''
    print(f'  💾 Disk free{tag}: {free:.1f} GB')
    if free < min_gb:
        raise RuntimeError(
            f'❌ Only {free:.1f} GB left on /kaggle/working — '
            f'aborting before crash (need {min_gb} GB).'
        )
    return free

def hf_file_exists(repo_id, filename):
    try:
        return filename in list(list_repo_files(repo_id))
    except Exception:
        return False

def hf_folder_exists(repo_id, prefix):
    try:
        return any(f.startswith(prefix) for f in list_repo_files(repo_id))
    except Exception:
        return False

def ensure_repo(repo_id):
    try:
        api.repo_info(repo_id=repo_id)
        print(f'  📁 Repo exists: {repo_id}')
    except Exception:
        api.create_repo(repo_id=repo_id, private=True)
        print(f'  📁 Created repo: {repo_id}')

def unwrap_clippable(model):
    try:
        from transformers.models.gemma4.modeling_gemma4 import Gemma4ClippableLinear
        ct = 0
        for n, m in list(model.named_modules()):
            if isinstance(m, Gemma4ClippableLinear):
                parts = n.split('.')
                setattr(model.get_submodule('.'.join(parts[:-1])), parts[-1], m.linear)
                ct += 1
        if ct:
            print(f'  Unwrapped {ct} Gemma4ClippableLinear modules')
    except Exception as e:
        print(f'  Unwrap skipped: {e}')

# ─── ENSURE REPOS EXIST ───────────────────────────────────────
ensure_repo(ADAPTER_REPO)
ensure_repo(GGUF_REPO)

print(f'\n✅ Setup complete.')
print(f'   GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'   Disk free : {disk_free_gb():.1f} GB')
print(f'   /dev/shm  : {shm_free_gb():.1f} GB')

✅ Logged in to HuggingFace
  📁 Repo exists: khedim/NLP-MINI-PROJECT-adapter
  📁 Repo exists: khedim/NLP-MINI-PROJECT

✅ Setup complete.
   GPU : Tesla T4
   Disk free : 20.1 GB
   /dev/shm  : 5.9 GB


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — STAGE 1: TRAIN
#
# Skip logic (in order):
#   1. adapter_config.json on ADAPTER_REPO  → skip entirely, download adapter
#   2. local checkpoint in MODEL_OUT        → resume from local
#   3. checkpoint-* folder on ADAPTER_REPO  → download latest, resume from it
#   4. nothing                              → train from scratch
#
# During training: each checkpoint is backed up to ADAPTER_REPO (non-blocking).
# After training:  final adapter is uploaded to ADAPTER_REPO.
# ═══════════════════════════════════════════════════════════════
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    Trainer, TrainingArguments, DataCollatorForLanguageModeling,
    TrainerCallback
)
from peft import LoraConfig, get_peft_model

print('=' * 60)
print('STAGE 1: TRAIN')
print('=' * 60)

# ── Check if training is already done ─────────────────────────
if hf_file_exists(ADAPTER_REPO, 'adapter_config.json'):
    print('✅ Final adapter found on HF Hub — skipping training.')
    if not os.path.exists(f'{MODEL_OUT}/adapter_config.json'):
        print(f'   Downloading adapter to {MODEL_OUT}...')
        snapshot_download(
            repo_id=ADAPTER_REPO,
            local_dir=MODEL_OUT,
            ignore_patterns=['checkpoint-*']
        )
    print('   Adapter ready locally.')
else:
    print('▶️  Starting / resuming training...')
    guard_disk(min_gb=3.0, label='before training')

    # ── Dataset ─────────────────────────────────────────────────
    print('\n  [1/4] Loading dataset...')
    ds = load_dataset(DATASET_ID, split='train', cache_dir=HF_CACHE)
    ds = ds.shuffle(seed=42).select(range(min(SAMPLE_SIZE, len(ds))))

    PREFIXES = [
        'Level: Age3-4 — Simple words.',
        'Level: Age5-6 — Short sentences.',
        'Level: Age7-8 — Moderate vocabulary.',
        'Level: Age9-10 — Longer sentences.',
        'Level: Age11-12 — Richer vocabulary.',
    ]
    def fmt(ex, i):
        pfx = PREFIXES[i % len(PREFIXES)]
        return {'text': (
            f'<start_of_turn>user\n{pfx}\n{ex.get("prompt", "")}'
            f'<end_of_turn>\n'
            f'<start_of_turn>model\n{ex.get("text", "")}'
            f'<end_of_turn>\n'
        )}

    ds = ds.map(fmt, with_indices=True, remove_columns=ds.column_names)
    print(f'  {len(ds)} samples formatted.')

    # ── Tokenizer + Model ────────────────────────────────────────
    print('\n  [2/4] Loading model...')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID, quantization_config=bnb_cfg, device_map='auto'
    )
    unwrap_clippable(model)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    for n, p in model.named_parameters():
        p.requires_grad = False
        if p.ndim == 1 and 'norm' in n.lower():
            p.data = p.data.to(torch.float32)

    lora_cfg = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=[
            'q_proj', 'k_proj', 'v_proj', 'o_proj',
            'gate_proj', 'up_proj', 'down_proj'
        ],
        lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # ── Tokenize dataset ─────────────────────────────────────────
    print('\n  [3/4] Tokenizing...')
    def tok(ex):
        o = tokenizer(
            ex['text'], truncation=True,
            max_length=MAX_SEQ_LEN, padding='max_length'
        )
        o['labels'] = o['input_ids'].copy()
        return o

    ds_tok = ds.map(tok, batched=True, remove_columns=['text'])
    split   = ds_tok.train_test_split(test_size=0.05, seed=42)
    train_ds, eval_ds = split['train'], split['test']
    print(f'  Train: {len(train_ds)}  |  Eval: {len(eval_ds)}')
    guard_disk(min_gb=3.0, label='after tokenize')

    # ── Determine resume checkpoint ───────────────────────────────
    resume = None

    # Priority 1: local checkpoint (same session)
    local_ckpts = sorted(glob.glob(f'{MODEL_OUT}/checkpoint-*'))
    if local_ckpts:
        resume = local_ckpts[-1]
        print(f'\n  ▶️  Resuming from local checkpoint: {resume}')

    # Priority 2: checkpoint on HF Hub (cross-session)
    if resume is None and hf_folder_exists(ADAPTER_REPO, 'checkpoint-'):
        print('\n  Found checkpoint on HF Hub. Downloading...')
        all_files = list(list_repo_files(ADAPTER_REPO))
        steps = sorted(set(
            int(f.split('/')[0].split('-')[1])
            for f in all_files
            if f.startswith('checkpoint-') and '/' in f
            and f.split('/')[0].split('-')[1].isdigit()
        ))
        if steps:
            ckpt_name = f'checkpoint-{steps[-1]}'
            print(f'  Downloading {ckpt_name} from HF Hub...')
            snapshot_download(
                repo_id=ADAPTER_REPO,
                local_dir=MODEL_OUT,
                allow_patterns=[f'{ckpt_name}/*'],
            )
            resume = f'{MODEL_OUT}/{ckpt_name}'
            print(f'  ✅ Resuming from HF checkpoint: {ckpt_name}')

    if resume is None:
        print('\n  No checkpoint found — training from scratch.')

    # ── HF checkpoint backup callback ────────────────────────────
    class HFCheckpointCallback(TrainerCallback):
        """Non-blocking upload of each checkpoint to HF Hub."""
        def on_save(self, args, state, control, **kwargs):
            if not state.is_world_process_zero:
                return
            step     = state.global_step
            ckpt_dir = f'{args.output_dir}/checkpoint-{step}'
            if not os.path.isdir(ckpt_dir):
                return
            try:
                print(f'\n  ☁️  Backing up checkpoint-{step} to HF Hub (async)...')
                api.upload_folder(
                    folder_path=ckpt_dir,
                    repo_id=ADAPTER_REPO,
                    path_in_repo=f'checkpoint-{step}',
                    commit_message=f'checkpoint-{step}',
                    run_as_future=True,   # non-blocking, training continues
                )
            except Exception as ex:
                print(f'  ⚠️  Checkpoint upload failed (safe to ignore, training continues): {ex}')

    # ── Disk guard callback ───────────────────────────────────────
    class DiskGuardCallback(TrainerCallback):
        """Abort training early if disk is almost full."""
        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step % 50 == 0:
                free = disk_free_gb()
                if free < 1.5:
                    print(f'\n  ❌ Disk critically low ({free:.1f} GB) — stopping training early.')
                    control.should_training_stop = True

    # ── Train ────────────────────────────────────────────────────
    print('\n  [4/4] Training...')
    train_args = TrainingArguments(
        output_dir=MODEL_OUT,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        optim='paged_adamw_8bit',
        save_steps=200,
        save_total_limit=1,           # keep only 1 checkpoint locally at a time
        logging_steps=20,
        learning_rate=2e-4,
        max_grad_norm=0.3,
        num_train_epochs=1,
        warmup_steps=10,
        lr_scheduler_type='cosine',
        fp16=True,
        gradient_checkpointing=True,
        eval_strategy='no',           # eval disabled — 256k vocab OOMs during eval
    )
    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=train_ds,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
        callbacks=[HFCheckpointCallback(), DiskGuardCallback()],
    )
    trainer.train(resume_from_checkpoint=resume)

    # ── Save final adapter ────────────────────────────────────────
    print('\n  Saving final adapter...')
    trainer.model.save_pretrained(MODEL_OUT)
    tokenizer.save_pretrained(MODEL_OUT)

    # Remove checkpoint folders to free disk before upload
    for ckpt in glob.glob(f'{MODEL_OUT}/checkpoint-*'):
        shutil.rmtree(ckpt, ignore_errors=True)

    guard_disk(min_gb=1.0, label='before adapter upload')

    print(f'  ☁️  Uploading final adapter to {ADAPTER_REPO}...')
    api.upload_folder(
        folder_path=MODEL_OUT,
        repo_id=ADAPTER_REPO,
        commit_message='Final LoRA adapter',
        ignore_patterns=['checkpoint-*'],
    )
    print('  ✅ Adapter uploaded.')

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    guard_disk(label='end of Stage 1')

print('\n✅ STAGE 1 COMPLETE')

STAGE 1: TRAIN
✅ Final adapter found on HF Hub — skipping training.
   Adapter ready locally.

✅ STAGE 1 COMPLETE


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — STAGE 2: MERGE TO DISK
# ═══════════════════════════════════════════════════════════════
import os, gc, shutil, torch

# Force override to Disk to prevent RAM crashes
MERGED_DIR = f'{BASE_DIR}/gemma_merged' 

print('=' * 60)
print('STAGE 2: MERGE TO DISK')
print('=' * 60)

# ── Check if merged model already exists ──────────────────────
if os.path.exists(f'{MERGED_DIR}/config.json'):
    print('✅ Merged model found on disk — skipping merge.')
else:
    # ── Lazy Imports to prevent circular dependency errors ────
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    # ── Ensure adapter is available locally ───────────────────
    if not os.path.exists(f'{MODEL_OUT}/adapter_config.json'):
        print('  Adapter not found locally — downloading from HF Hub...')
        snapshot_download(
            repo_id=ADAPTER_REPO,
            local_dir=MODEL_OUT,
            ignore_patterns=['checkpoint-*'],
        )

    # ── Step 1: Wipe HF Cache to free up Disk Space ───────────
    # We must do this BEFORE loading the base model so the 
    # 10GB FP16 download doesn't hit the 20GB disk limit.
    print('\n  [1/4] Clearing HF cache to make room on disk...')
    shutil.rmtree(HF_CACHE, ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()
    guard_disk(min_gb=10.0, label='after cache wipe')

    # ── Step 2: Load Base Model ───────────────────────────────
    print('\n  [2/4] Loading base model (FP16)...')
    # Use low_cpu_mem_usage to keep the RAM footprint small
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16,
        device_map="cpu",
        low_cpu_mem_usage=True 
    )
    unwrap_clippable(base)

    # ── Step 3: Merge LoRA Weights ────────────────────────────
    print('  [3/4] Merging LoRA weights...')
    model = PeftModel.from_pretrained(base, MODEL_OUT)
    merged = model.merge_and_unload()
    
    # Immediately clear memory-heavy objects
    del base, model
    gc.collect()

    # ── Step 4: Save Merged Model to Disk ─────────────────────
    print(f'  [4/4] Saving merged model to {MERGED_DIR}...')
    os.makedirs(MERGED_DIR, exist_ok=True)
    
    # Sharding to 2GB ensures we don't hit "file too large" errors 
    # or buffer overflows during the write process.
    merged.save_pretrained(
        MERGED_DIR, 
        safe_serialization=True, 
        max_shard_size="2GB"
    )
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)
    tokenizer.save_pretrained(MERGED_DIR)

    # Final cleanup
    del merged, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

    print(f'\n✅ STAGE 2 COMPLETE')
    print(f'   Disk used: {shutil.disk_usage(BASE_DIR).used / 1e9:.1f} GB')
    print(f'   RAM free: {shm_free_gb():.1f} GB')

STAGE 2: MERGE TO DISK

  [1/4] Clearing HF cache to make room on disk...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


  💾 Disk free [after cache wipe]: 20.1 GB

  [2/4] Loading base model (FP16)...


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

  Unwrapped 232 Gemma4ClippableLinear modules
  [3/4] Merging LoRA weights...
  [4/4] Saving merged model to /kaggle/working/gemma_merged...


Writing model shards:   0%|          | 0/4 [00:00<?, ?it/s]


✅ STAGE 2 COMPLETE
   Disk used: 11.1 GB
   RAM free: 5.9 GB


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — STAGE 3: CONVERT TO GGUF + QUANTIZE
#
# /dev/shm (merged FP16)  →  disk (FP16 GGUF)  →  disk (Q4_K_M)
#
# Each intermediate is deleted the moment it's no longer needed,
# so only one large file exists on disk at a time.
#
# Skip logic: checks GGUF_REPO for the final file first.
#
# NOTE: /dev/shm is cleared on session restart. If you restarted
# the kernel, re-run Cell 4 before this cell.
# ═══════════════════════════════════════════════════════════════
import subprocess

print('=' * 60)
print('STAGE 3: CONVERT & QUANTIZE')
print('=' * 60)

GGUF_FILENAME    = 'gemma-e2b-Q4_K_M.gguf'
QUANTIZE_BIN     = f'{LLAMA_DIR}/build/bin/llama-quantize'
CONVERT_PY       = f'{LLAMA_DIR}/convert_hf_to_gguf.py'

# ── Skip if already on HF ─────────────────────────────────────
if hf_file_exists(GGUF_REPO, GGUF_FILENAME):
    print(f'✅ STAGE 3 SKIPPED — {GGUF_FILENAME} already on HF Hub.')
else:
    # ── Guard: merged model must exist in RAM ─────────────────
    if not os.path.exists(f'{MERGED_DIR}/config.json'):
        raise RuntimeError(
            '❌ Merged model not found in /dev/shm.\n'
            '   /dev/shm is wiped on kernel restart.\n'
            '   → Re-run Cell 4 (Stage 2) first.'
        )

    guard_disk(min_gb=5.5, label='before GGUF conversion')  # needs ~4.5 GB for FP16 GGUF

    # ── Build llama.cpp (skipped if already built) ────────────
    if not os.path.exists(QUANTIZE_BIN):
        print('\n  Building llama.cpp (one-time, ~5 min)...')
        if not os.path.exists(LLAMA_DIR):
            subprocess.run(
                ['git', 'clone', '--depth', '1',
                 'https://github.com/ggerganov/llama.cpp.git', LLAMA_DIR],
                check=True
            )
        subprocess.run(['cmake', '-B', 'build'], cwd=LLAMA_DIR, check=True)
        subprocess.run(
            ['cmake', '--build', 'build', '--config', 'Release', '-j', '4'],
            cwd=LLAMA_DIR, check=True
        )
        subprocess.run(
            ['pip', 'install', '-q', f'{LLAMA_DIR}/gguf-py'], check=True
        )
        print('  ✅ llama.cpp built.')
    else:
        print('  ✅ llama.cpp already built — skipping compile.')
        subprocess.run(['pip', 'install', '-q', f'{LLAMA_DIR}/gguf-py'], check=True)

    # ── Convert /dev/shm → FP16 GGUF ─────────────────────────
    # Delete any stale FP16 GGUF from a previous failed attempt
    if os.path.exists(FP16_GGUF):
        os.remove(FP16_GGUF)

    print(f'\n  Converting {MERGED_DIR} → FP16 GGUF...')
    result = subprocess.run([
        'python', CONVERT_PY, MERGED_DIR,
        '--outfile', FP16_GGUF, '--outtype', 'f16'
    ])
    if result.returncode != 0 or not os.path.exists(FP16_GGUF):
        raise RuntimeError(
            '❌ GGUF conversion failed.\n'
            '   Merged model is still safe in /dev/shm — check the error above.'
        )

    fp16_size = os.path.getsize(FP16_GGUF) / 1e9
    print(f'  ✅ FP16 GGUF: {fp16_size:.2f} GB')

    # Free /dev/shm immediately — we no longer need the safetensors
    print('  Freeing /dev/shm (~9 GB RAM)...')
    shutil.rmtree(MERGED_DIR, ignore_errors=True)
    gc.collect()
    guard_disk(label='after /dev/shm free')

    # ── Quantize FP16 → Q4_K_M ───────────────────────────────
    print('\n  Quantizing to Q4_K_M...')
    result = subprocess.run([QUANTIZE_BIN, FP16_GGUF, Q4_GGUF, 'Q4_K_M'])

    # Delete FP16 GGUF immediately to free ~4.5 GB
    if os.path.exists(FP16_GGUF):
        os.remove(FP16_GGUF)

    if result.returncode != 0 or not os.path.exists(Q4_GGUF):
        raise RuntimeError('❌ Quantization failed. Check output above.')

    q4_size = os.path.getsize(Q4_GGUF) / 1e9
    print(f'  ✅ Q4_K_M GGUF: {q4_size:.2f} GB')
    guard_disk(label='end of Stage 3')

print('\n✅ STAGE 3 COMPLETE')

STAGE 3: CONVERT & QUANTIZE
  💾 Disk free [before GGUF conversion]: 9.8 GB
  ✅ llama.cpp already built — skipping compile.

  Converting /kaggle/working/gemma_merged → FP16 GGUF...


INFO:hf-to-gguf:Loading model: gemma_merged
INFO:numexpr.utils:NumExpr defaulting to 4 threads.
INFO:hf-to-gguf:Model architecture: Gemma4ForConditionalGeneration
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,                 torch.float32 --> F32, shape = {256}
INFO:hf-to-gguf:per_layer_token_embd.weight,       torch.float16 --> F16, shape = {8960, 262144}
INFO:hf-to-gguf:token_embd.weight,                 torch.float16 --> F16, shape = {1536, 262144}
INFO:hf-to-gguf:blk.0.attn_norm.weight,            torch.flo

  ✅ FP16 GGUF: 9.27 GB
  Freeing /dev/shm (~9 GB RAM)...
  💾 Disk free [after /dev/shm free]: 10.8 GB

  Quantizing to Q4_K_M...


PermissionError: [Errno 13] Permission denied: '/kaggle/working/llama.cpp/build/bin/llama-quantize'

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — STAGE 4: UPLOAD FINAL GGUF
#
# Uploads Q4_K_M GGUF to GGUF_REPO, then cleans up.
# Safe to re-run — skips if already uploaded.
# ═══════════════════════════════════════════════════════════════
print('=' * 60)
print('STAGE 4: UPLOAD')
print('=' * 60)

GGUF_FILENAME = 'gemma-e2b/gemma-e2b-Q4_K_M.gguf'

if hf_file_exists(GGUF_REPO, GGUF_FILENAME):
    print(f'✅ STAGE 4 SKIPPED — {GGUF_FILENAME} already on HF Hub.')
else:
    if not os.path.exists(Q4_GGUF):
        raise RuntimeError(
            f'❌ {Q4_GGUF} not found.\n'
            '   → Re-run Cell 5 (Stage 3) first.'
        )

    size_gb = os.path.getsize(Q4_GGUF) / 1e9
    print(f'  Uploading {size_gb:.2f} GB to {GGUF_REPO}...')

    api.upload_file(
        path_or_fileobj=Q4_GGUF,
        path_in_repo=GGUF_FILENAME,
        repo_id=GGUF_REPO,
        commit_message='Fine-tuned Gemma 4 E2B Q4_K_M GGUF',
    )
    print('  ✅ Upload complete.')

    # Clean up local GGUF
    os.remove(Q4_GGUF)
    guard_disk(label='final')

print('\n' + '=' * 60)
print(f'🎉 ALL DONE!')
print(f'   GGUF : https://huggingface.co/{GGUF_REPO}')
print(f'   LoRA : https://huggingface.co/{ADAPTER_REPO}')
print('=' * 60)